# Genetic Algorithm for the Dynamic Traveling Salesman Problem
### How every number in the report was calculated (Santa Rosa, Laguna edition)

This notebook rebuilds the whole project step by step, so you can **run it, change it and explain it**.

| Step | What you do | Where it appears in the report |
|---|---|---|
| 1 | See why the TSP is hard | Introduction |
| 2 | Build the Genetic Algorithm from scratch | Section 4.1 |
| 3 | Build the comparison algorithms | Sections 3 and 4 |
| 4 | Test correctness with explicit **test cases** (expected vs actual) | Section 6 |
| 5 | Static comparison (Experiment 1) | Table 1, Figure 1 |
| 6 | Dynamic map changes (Experiment 2) | Table 2, Figure 2 |
| 7 | Santa Rosa, Laguna scenario | Santa Rosa simulation |
| 8 | Five **use cases** with equal time budgets and measured results | Section 8, Figure 3 |

**How to run:** `Runtime > Run all`. The full run takes roughly 8 to 12 minutes. Set `FAST = True` in the next cell for a
shorter run (about 3 minutes) with noisier numbers.

**Important:** the time-budget scenarios (Step 8) depend on your computer's speed, and several methods use random numbers, so
your results will be close to the report but **not identical**. The patterns (who wins, by roughly how much) are what matter.

In [ ]:
FAST = False   # True = shorter runs (noisier numbers); False = full-length runs like the report

import os, math, random, time
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.makedirs("output", exist_ok=True)
print("Ready. FAST =", FAST)

## Step 1. Why the problem is hard

A **tour** visits every stop once and returns to the start. Its **length** is the sum of the distances between consecutive stops
(including the way back). For *n* stops there are (n-1)!/2 different tours, which explodes quickly.

In [ ]:
# Tour length for a tiny example: four corners of a unit square
pts = [(0, 0), (0, 1), (1, 1), (1, 0)]

def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def tour_len(order):
    return sum(dist(pts[order[i]], pts[order[(i + 1) % len(order)]]) for i in range(len(order)))

print("Around the square   :", tour_len([0, 1, 2, 3]))      # 4.0
print("Crossing diagonals  :", round(tour_len([0, 2, 1, 3]), 3))  # 2 + 2*sqrt(2) = 4.828

# How many distinct tours exist?
for n in (5, 10, 15, 20, 100):
    count = math.factorial(n - 1) // 2
    print(f"{n:>3} stops -> {count:,} tours" if n <= 20 else f"{n:>3} stops -> about 10^{len(str(count)) - 1} tours")

## Step 2. The Genetic Algorithm, written from scratch

**Genetics metaphor:** a stop is a *gene*, a tour is a *chromosome*, a set of tours is a *population*.

Each generation:
1. **Selection (tournament):** pick 5 random tours and keep the shortest one as a parent.
2. **Order Crossover (OX):** copy a slice from parent 1, fill the rest in the order stops appear in parent 2. The child is always a valid tour.
3. **Mutation:** reverse a segment (70%) or swap two stops (30%).
4. **2-opt refinement:** for a small share of children, remove crossing roads (this makes it a *memetic* GA).
5. **Elitism:** the 2 best tours always survive.

**Dynamic support:** `add_city` inserts a new stop into every tour at its cheapest position, `remove_city` deletes a stop from every tour,
and `set_distances` swaps in a new cost matrix (traffic, floods). Run the next cell to write the file `ga.py`.

In [ ]:
%%writefile ga.py
"""
ga.py - Genetic Algorithm for the Traveling Salesman Problem (from scratch).

Representation : a tour is a permutation of city indices [0..n-1].
Selection      : tournament selection (size k).
Crossover      : Order Crossover (OX) - preserves relative order, always valid.
Local search   : bounded 2-opt on a fraction of children (memetic GA).
Mutation       : inversion (reverses a segment) or swap, chosen at random.
Elitism        : the best `elite` tours survive unchanged every generation.

Dynamic support: cities can be added/removed WHILE the population evolves.
Instead of restarting, every existing tour is repaired (cheapest insertion for
a new city, deletion + relabel for a removed one), so earlier progress is kept.

Only the Python standard library is used (no library calls for core logic).
"""
import math
import random


def euclid(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def build_matrix(points, dist_fn=euclid):
    """Full symmetric distance matrix, O(n^2) time and space."""
    n = len(points)
    D = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = dist_fn(points[i], points[j])
            D[i][j] = D[j][i] = d
    return D


def tour_length(tour, D):
    """Length of the closed tour, O(n)."""
    total = 0.0
    prev = tour[-1]
    for c in tour:
        total += D[prev][c]
        prev = c
    return total


def order_crossover(p1, p2, rng):
    """OX: copy a slice from p1, fill the rest in the order they appear in p2."""
    n = len(p1)
    a, b = sorted(rng.sample(range(n), 2)) if n > 1 else (0, 0)
    child = [-1] * n
    child[a:b + 1] = p1[a:b + 1]
    used = set(child[a:b + 1])
    pos = (b + 1) % n
    for gene in p2[b + 1:] + p2[:b + 1]:
        if gene not in used:
            child[pos] = gene
            pos = (pos + 1) % n
    return child


def mutate(tour, rng):
    n = len(tour)
    if n < 3:
        return
    i, j = sorted(rng.sample(range(n), 2))
    if rng.random() < 0.7:          # inversion (a 2-opt style move)
        tour[i:j + 1] = reversed(tour[i:j + 1])
    else:                            # swap
        tour[i], tour[j] = tour[j], tour[i]


class GeneticTSP:
    def __init__(self, points, pop_size=150, elite=2, tournament_k=5,
                 crossover_rate=0.9, mutation_rate=0.25, seed=None,
                 greedy_seed=True, local_rate=0.15, dist_fn=euclid):
        self.rng = random.Random(seed)
        self.points = list(points)
        self.dist_fn = dist_fn          # e.g. Haversine for GPS points
        self.D = build_matrix(self.points, dist_fn)
        self.pop_size = pop_size
        self.elite = elite
        self.k = tournament_k
        self.cx = crossover_rate
        self.mut = mutation_rate
        self.local_rate = local_rate   # fraction of children refined by 2-opt
        self.generation = 0
        self.evaluations = 0
        n = len(self.points)
        self.pop = []
        for _ in range(pop_size):
            t = list(range(n))
            self.rng.shuffle(t)
            self.pop.append(t)
        if greedy_seed and n > 2:           # one nearest-neighbour tour seeds quality
            self.pop[0] = self._nearest_neighbour(self.rng.randrange(n))
        self._score()

    # ---- helpers -------------------------------------------------------
    def _nearest_neighbour(self, start):
        n = len(self.points)
        left = set(range(n))
        left.remove(start)
        tour = [start]
        while left:
            last = tour[-1]
            nxt = min(left, key=lambda c: self.D[last][c])
            left.remove(nxt)
            tour.append(nxt)
        return tour

    def _score(self):
        self.fit = [tour_length(t, self.D) for t in self.pop]
        self.evaluations += len(self.pop)

    def _tournament(self):
        best = None
        for _ in range(self.k):
            i = self.rng.randrange(len(self.pop))
            if best is None or self.fit[i] < self.fit[best]:
                best = i
        return self.pop[best]

    def _two_opt(self, t, max_passes=3):
        """Bounded first-improvement 2-opt (memetic step), O(n^2) per pass."""
        D, n = self.D, len(t)
        for _ in range(max_passes):
            improved = False
            for i in range(n - 1):
                a, b = t[i], t[i + 1]
                for j in range(i + 2, n):
                    if i == 0 and j == n - 1:
                        continue
                    c, d = t[j], t[(j + 1) % n]
                    if D[a][c] + D[b][d] < D[a][b] + D[c][d] - 1e-12:
                        t[i + 1:j + 1] = reversed(t[i + 1:j + 1])
                        a, b = t[i], t[i + 1]
                        improved = True
            if not improved:
                break
        return t

    # ---- evolution -----------------------------------------------------
    def step(self):
        """Advance one generation."""
        order = sorted(range(len(self.pop)), key=self.fit.__getitem__)
        new_pop = [self.pop[i][:] for i in order[:self.elite]]
        while len(new_pop) < self.pop_size:
            p1, p2 = self._tournament(), self._tournament()
            child = (order_crossover(p1, p2, self.rng)
                     if self.rng.random() < self.cx else p1[:])
            if self.rng.random() < self.mut:
                mutate(child, self.rng)
            if self.rng.random() < self.local_rate and len(child) > 3:
                self._two_opt(child)
            new_pop.append(child)
        self.pop = new_pop
        self._score()
        self.generation += 1

    def run(self, generations):
        history = []
        for _ in range(generations):
            self.step()
            history.append(self.best()[1])
        return history

    def best(self):
        i = min(range(len(self.pop)), key=self.fit.__getitem__)
        return self.pop[i][:], self.fit[i]

    # ---- dynamic changes ----------------------------------------------
    def add_city(self, point):
        """Add a city; insert it at its cheapest position in every tour."""
        new = len(self.points)
        self.points.append(point)
        for row in self.D:
            row.append(0.0)
        self.D.append([0.0] * (new + 1))
        for i in range(new):
            d = self.dist_fn(self.points[i], point)
            self.D[i][new] = self.D[new][i] = d
        D = self.D
        for t in self.pop:
            best_pos, best_cost = 0, float("inf")
            for p in range(len(t)):
                a, b = t[p - 1], t[p]
                cost = D[a][new] + D[new][b] - D[a][b]
                if cost < best_cost:
                    best_cost, best_pos = cost, p
            t.insert(best_pos, new)
        self._score()
        return new

    def remove_city(self, idx):
        """Remove city idx; delete from every tour and relabel higher indices."""
        del self.points[idx]
        del self.D[idx]
        for row in self.D:
            del row[idx]
        for t in self.pop:
            t.remove(idx)
            for p, c in enumerate(t):
                if c > idx:
                    t[p] = c - 1
        self._score()

    def set_distances(self, D):
        """Replace the whole cost matrix (traffic, closures...) and re-score.
        Tours stay valid; only their lengths change, so the population keeps
        its structure and simply re-adapts to the new costs."""
        self.D = D
        self._score()

### See the operators work on a tiny example

In [ ]:
import random
from ga import GeneticTSP, order_crossover, mutate, build_matrix, tour_length

rng = random.Random(1)
p1 = [0, 1, 2, 3, 4, 5, 6, 7]
p2 = [3, 7, 5, 1, 6, 0, 2, 4]
child = order_crossover(p1, p2, rng)
print("parent 1 :", p1)
print("parent 2 :", p2)
print("child    :", child, "(always a valid permutation:", sorted(child) == sorted(p1), ")")

t = p1[:]
mutate(t, rng)
print("mutated  :", t)

# A short evolution run on 30 random cities, printing the best length every 20 generations
rng = random.Random(5)
cities = [(rng.random() * 100, rng.random() * 100) for _ in range(30)]
ga = GeneticTSP(cities, pop_size=80, seed=1)
history = []
for g in range(100):
    ga.step()
    history.append(ga.best()[1])
    if g % 20 == 0:
        print(f"generation {g:3d}  best length {history[-1]:.1f}")
plt.plot(history); plt.xlabel("generation"); plt.ylabel("best tour length"); plt.title("GA convergence on 30 random cities")
plt.grid(alpha=.3); plt.show()

## Step 3. The comparison algorithms

| Algorithm | Idea | Cost |
|---|---|---|
| Nearest Neighbour | always drive to the closest unvisited stop | O(n^2), instant |
| 2-opt | keep uncrossing pairs of roads until none help | O(n^2) per pass |
| Simulated Annealing (Kirkpatrick et al., 1983) | accept worse moves with probability exp(-delta/T), lower T over time | one 2-opt move per step |
| Tabu Search (Glover, 1989) | take the best sampled move even if worse, but forbid recent moves | sampled neighbourhood per step |
| Held-Karp (exact) | dynamic programming over subsets of stops | O(n^2 * 2^n), only for n up to about 13 |

Held-Karp gives the **true optimum**, so we can measure how far each heuristic is from perfect on small problems.

In [ ]:
%%writefile baselines.py
"""
baselines.py - Comparison algorithms for the research phase.

nearest_neighbour : O(n^2) greedy construction (quality baseline)
two_opt           : O(n^2) per pass local search
simulated_annealing : probabilistic hill climbing with a cooling schedule
tabu_search       : 2-opt neighbourhood with a short-term memory
held_karp         : exact dynamic programming, O(n^2 * 2^n) - small n only,
                    used by the tests as ground truth.
"""
import math
import random
from ga import tour_length


def nearest_neighbour(D, start=0):
    n = len(D)
    left = set(range(n))
    left.remove(start)
    tour = [start]
    while left:
        last = tour[-1]
        nxt = min(left, key=lambda c: D[last][c])
        left.remove(nxt)
        tour.append(nxt)
    return tour


def two_opt(tour, D):
    tour = tour[:]
    n = len(tour)
    improved = True
    while improved:
        improved = False
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:
                    continue
                a, b = tour[i], tour[i + 1]
                c, d = tour[j], tour[(j + 1) % n]
                if D[a][c] + D[b][d] < D[a][b] + D[c][d] - 1e-12:
                    tour[i + 1:j + 1] = reversed(tour[i + 1:j + 1])
                    improved = True
    return tour


def simulated_annealing(D, iterations=200000, t0=None, alpha=None, seed=None):
    """Geometric cooling; neighbour = random segment reversal (2-opt move)."""
    rng = random.Random(seed)
    n = len(D)
    tour = nearest_neighbour(D)
    cur = tour_length(tour, D)
    best, best_len = tour[:], cur
    t0 = t0 or cur / n
    t_end = t0 * 1e-4
    alpha = alpha or (t_end / t0) ** (1.0 / iterations)
    T = t0
    history = []
    for it in range(iterations):
        i, j = sorted(rng.sample(range(n), 2))
        if i == 0 and j == n - 1:
            continue
        a, b = tour[i - 1], tour[i]
        c, d = tour[j], tour[(j + 1) % n]
        delta = D[a][c] + D[b][d] - D[a][b] - D[c][d]
        if delta < 0 or rng.random() < math.exp(-delta / T):
            tour[i:j + 1] = reversed(tour[i:j + 1])
            cur += delta
            if cur < best_len - 1e-12:
                best, best_len = tour[:], cur
        T *= alpha
        if it % 1000 == 0:
            history.append(best_len)
    return best, tour_length(best, D), history


def tabu_search(D, iterations=300, tenure=15, seed=None):
    """Best-improvement over sampled 2-opt moves, with a tabu list of edges."""
    rng = random.Random(seed)
    n = len(D)
    tour = nearest_neighbour(D)
    best, best_len = tour[:], tour_length(tour, D)
    cur_len = best_len
    tabu = {}
    history = []
    for it in range(iterations):
        move, move_delta = None, float("inf")
        for _ in range(min(200, n * n)):
            i, j = sorted(rng.sample(range(n), 2))
            if i == 0 and j == n - 1:
                continue
            a, b = tour[i - 1], tour[i]
            c, d = tour[j], tour[(j + 1) % n]
            delta = D[a][c] + D[b][d] - D[a][b] - D[c][d]
            key = (min(a, c), max(a, c))
            is_tabu = tabu.get(key, -1) > it
            if is_tabu and cur_len + delta >= best_len:   # aspiration criterion
                continue
            if delta < move_delta:
                move, move_delta = (i, j, key), delta
        if move is None:
            continue
        i, j, key = move
        tour[i:j + 1] = reversed(tour[i:j + 1])
        cur_len += move_delta
        tabu[key] = it + tenure
        if cur_len < best_len - 1e-12:
            best, best_len = tour[:], cur_len
        history.append(best_len)
    return best, tour_length(best, D), history


def held_karp(D):
    """Exact optimum length via bitmask DP. Only use for n <= ~13."""
    n = len(D)
    if n < 2:
        return 0.0
    INF = float("inf")
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1][0] = 0.0
    for mask in range(1 << n):
        if not mask & 1:
            continue
        for last in range(n):
            cur = dp[mask][last]
            if cur == INF or not (mask >> last) & 1:
                continue
            for nxt in range(n):
                if (mask >> nxt) & 1:
                    continue
                nm = mask | (1 << nxt)
                v = cur + D[last][nxt]
                if v < dp[nm][nxt]:
                    dp[nm][nxt] = v
    full = (1 << n) - 1
    return min(dp[full][last] + D[last][0] for last in range(1, n))

In [ ]:
from baselines import nearest_neighbour, two_opt, held_karp, simulated_annealing, tabu_search

rng = random.Random(3)
small = [(rng.random() * 100, rng.random() * 100) for _ in range(10)]
D = build_matrix(small)

opt = held_karp(D)                                  # exact optimum
nn_len = tour_length(nearest_neighbour(D), D)
opt2_len = tour_length(two_opt(nearest_neighbour(D), D), D)
ga = GeneticTSP(small, pop_size=60, seed=2); ga.run(80)

print(f"exact optimum (Held-Karp) : {opt:.2f}")
print(f"nearest neighbour         : {nn_len:.2f}  ({100*(nn_len-opt)/opt:.1f}% above optimum)")
print(f"2-opt from nearest nbr    : {opt2_len:.2f}  ({100*(opt2_len-opt)/opt:.1f}% above optimum)")
print(f"genetic algorithm         : {ga.best()[1]:.2f}  ({100*(ga.best()[1]-opt)/opt:.2f}% above optimum)")

## Step 4. Is the GA correct? Automated tests

The report says *seven automated tests pass*. They check:
- Order Crossover always produces a valid permutation (800 random trials).
- Tour length of a unit square is exactly 4.
- **GA matches the exact Held-Karp optimum on five random 10-city instances.**
- GA finds the exact perimeter of a 16-city circle.
- On 200 cities the GA returns valid tours, its best length never increases (elitism), and it beats a greedy tour.
- 2-opt improves a random tour.
- After adding and removing cities, every tour in the population is still valid.

In [ ]:
%%writefile test_tsp.py
"""Run with:  python -m unittest discover -s tests -v   (from the project root)"""
import os
import random
import sys
import unittest

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from ga import GeneticTSP, build_matrix, tour_length, order_crossover
from baselines import held_karp, nearest_neighbour, two_opt


def rand_points(n, seed):
    r = random.Random(seed)
    return [(r.random() * 100, r.random() * 100) for _ in range(n)]


def is_perm(t, n):
    return sorted(t) == list(range(n))


class TestOperators(unittest.TestCase):
    def test_crossover_always_valid(self):
        rng = random.Random(1)
        for n in (3, 5, 20, 100):
            for _ in range(200):
                a, b = list(range(n)), list(range(n))
                rng.shuffle(a)
                rng.shuffle(b)
                self.assertTrue(is_perm(order_crossover(a, b, rng), n))

    def test_square_tour_length(self):
        D = build_matrix([(0, 0), (0, 1), (1, 1), (1, 0)])
        self.assertAlmostEqual(tour_length([0, 1, 2, 3], D), 4.0)
        self.assertAlmostEqual(tour_length([0, 2, 1, 3], D), 2 + 2 * 2 ** 0.5)


class TestSmallOptimal(unittest.TestCase):
    """GA must reach the exact optimum (Held-Karp) on small instances."""

    def test_matches_exact_optimum(self):
        for seed in range(5):
            pts = rand_points(10, seed)
            D = build_matrix(pts)
            opt = held_karp(D)
            ga = GeneticTSP(pts, pop_size=100, seed=seed)
            ga.run(150)
            tour, length = ga.best()
            self.assertTrue(is_perm(tour, 10))
            self.assertAlmostEqual(length, opt, places=6,
                                   msg=f"seed {seed}: GA {length} vs opt {opt}")

    def test_known_circle(self):
        import math
        n = 16
        pts = [(math.cos(2 * math.pi * i / n), math.sin(2 * math.pi * i / n))
               for i in range(n)]
        random.Random(3).shuffle(pts)
        ga = GeneticTSP(pts, pop_size=150, seed=3)
        ga.run(300)
        perimeter = n * 2 * math.sin(math.pi / n)
        self.assertAlmostEqual(ga.best()[1], perimeter, places=6)


class TestLarge(unittest.TestCase):
    def test_200_cities_valid_and_better_than_greedy(self):
        pts = rand_points(200, 42)
        D = build_matrix(pts)
        nn_len = tour_length(nearest_neighbour(D), D)
        ga = GeneticTSP(pts, pop_size=120, seed=42, greedy_seed=False)
        hist = ga.run(400)
        tour, length = ga.best()
        self.assertTrue(is_perm(tour, 200))
        self.assertLessEqual(hist[-1], hist[0])          # elitism => monotone
        self.assertTrue(all(hist[i + 1] <= hist[i] + 1e-9
                            for i in range(len(hist) - 1)))
        self.assertLess(length, nn_len * 1.15)           # competitive with greedy

    def test_two_opt_improves(self):
        pts = rand_points(80, 7)
        D = build_matrix(pts)
        t = list(range(80))
        random.Random(7).shuffle(t)
        self.assertLess(tour_length(two_opt(t, D), D), tour_length(t, D))


class TestDynamic(unittest.TestCase):
    def test_add_and_remove_keep_population_valid(self):
        pts = rand_points(30, 5)
        ga = GeneticTSP(pts, pop_size=60, seed=5)
        ga.run(30)
        for k in range(5):
            ga.add_city((random.random() * 100, random.random() * 100))
            ga.run(5)
        ga.remove_city(3)
        ga.remove_city(0)
        ga.run(10)
        n = len(ga.points)
        self.assertEqual(n, 33)
        self.assertTrue(all(is_perm(t, n) for t in ga.pop))
        tour, length = ga.best()
        self.assertAlmostEqual(length, tour_length(tour, build_matrix(ga.points)))


if __name__ == "__main__":
    unittest.main()

In [ ]:
os.makedirs("tests", exist_ok=True)
os.replace("test_tsp.py", "tests/test_tsp.py")
!python -m unittest discover -s tests -v

### Test cases with expected vs actual results

The unit tests above pass or fail silently. This table makes each **test case** explicit: an ID, what is being tested, the **expected**
result, the **actual** result computed live, and PASS or FAIL.

In [ ]:
import math, random
from ga import GeneticTSP, build_matrix, tour_length, order_crossover
from baselines import held_karp

results = []
def check(tc, what, expected, actual, ok):
    results.append((tc, what, expected, actual, "PASS" if ok else "FAIL"))

# TC-01 and TC-02: tour length on a unit square
sq = build_matrix([(0, 0), (0, 1), (1, 1), (1, 0)])
a = tour_length([0, 1, 2, 3], sq)
check("TC-01", "Square, tour around the edge", "4.000", f"{a:.3f}", abs(a - 4) < 1e-9)
b = tour_length([0, 2, 1, 3], sq)
check("TC-02", "Square, tour with crossing diagonals", "2+2*sqrt(2) = 4.828", f"{b:.3f}", abs(b - (2 + 2 * math.sqrt(2))) < 1e-9)

# TC-03: Order Crossover always returns a valid tour
rng = random.Random(1); ok = True
for _ in range(200):
    p, q = list(range(20)), list(range(20)); rng.shuffle(p); rng.shuffle(q)
    ok &= sorted(order_crossover(p, q, rng)) == list(range(20))
check("TC-03", "Order Crossover, 200 random parent pairs (20 stops)", "every child valid", "all valid" if ok else "invalid child found", ok)

# TC-04: GA reaches the exact optimum on small random problems
gaps = []
for seed in range(3):
    r = random.Random(seed); pts = [(r.random() * 100, r.random() * 100) for _ in range(10)]
    D = build_matrix(pts); opt = held_karp(D)
    g = GeneticTSP(pts, pop_size=100, seed=seed); g.run(150)
    gaps.append(g.best()[1] - opt)
check("TC-04", "GA vs exact Held-Karp, 3 random 10-stop maps", "gap = 0", f"max gap {max(gaps):.1e}", max(gaps) < 1e-6)

# TC-05: GA finds the known optimum of a circle (the perimeter of the polygon)
n = 16
circle = [(math.cos(2 * math.pi * i / n), math.sin(2 * math.pi * i / n)) for i in range(n)]
random.Random(3).shuffle(circle)
g = GeneticTSP(circle, pop_size=150, seed=3); g.run(300)
perimeter = n * 2 * math.sin(math.pi / n)
check("TC-05", "16 points on a circle (shuffled)", f"perimeter {perimeter:.4f}", f"{g.best()[1]:.4f}", abs(g.best()[1] - perimeter) < 1e-6)

# TC-06: adding and removing stops keeps every tour valid
r = random.Random(5); pts = [(r.random() * 100, r.random() * 100) for _ in range(30)]
g = GeneticTSP(pts, pop_size=60, seed=5); g.run(30)
for _ in range(5):
    g.add_city((r.random() * 100, r.random() * 100)); g.run(5)
g.remove_city(3); g.remove_city(0); g.run(10)
n_now = len(g.points)
valid = all(sorted(t) == list(range(n_now)) for t in g.pop)
same_len = abs(g.best()[1] - tour_length(g.best()[0], build_matrix(g.points))) < 1e-9
check("TC-06", "Start 30 stops, add 5, remove 2", "33 stops, all tours valid, length consistent", f"{n_now} stops, valid={valid}, consistent={same_len}", n_now == 33 and valid and same_len)

# TC-07: elitism means the best length never gets worse
r = random.Random(7); pts = [(r.random() * 100, r.random() * 100) for _ in range(50)]
g = GeneticTSP(pts, pop_size=60, seed=7); h = g.run(100)
mono = all(h[i + 1] <= h[i] + 1e-9 for i in range(len(h) - 1))
check("TC-07", "50 stops, 100 generations", "best length never increases", "never increased" if mono else "increased", mono)

# TC-08: a cost change (flood: links to stop 5 cost 3x) is picked up correctly
r = random.Random(9); pts = [(r.random() * 100, r.random() * 100) for _ in range(20)]
g = GeneticTSP(pts, pop_size=60, seed=9); g.run(20)
D2 = [[d * (3 if (i != j and 5 in (i, j)) else 1) for j, d in enumerate(row)] for i, row in enumerate(g.D)]
before = g.best()[1]; g.set_distances(D2)
t, L = g.best()
check("TC-08", "Flood: all links to stop 5 cost 3x", "length re-scored under new costs", f"{before:.1f} -> {L:.1f}", abs(L - tour_length(t, D2)) < 1e-9 and L >= before - 1e-9)

print(f"{'ID':<6}{'What is tested':<52}{'Expected':<48}{'Actual':<44}Result")
print("-" * 158)
for row in results:
    print(f"{row[0]:<6}{row[1]:<52}{row[2]:<48}{row[3]:<44}{row[4]}")
print(f"\n{sum(r[4] == 'PASS' for r in results)} of {len(results)} test cases passed")

## Step 5. Experiment 1: static comparison (Table 1, Figure 1)

For n = 30, 60 and 100 random cities in a 100 x 100 square, each algorithm solves the same instances (3 per size).
The table reports the **mean tour length** (lower is better) and the running time.

This comparison is **not equal-time**: each method used its own settings (GA: 400 generations of 80 tours; SA: 150,000 steps;
Tabu: 250 iterations). That is why Step 8 repeats the comparison with equal time budgets.

In [ ]:
%%writefile simulate.py
"""
simulate.py - Experiments and live simulation for the GA-TSP project.

  python simulate.py            -> runs everything, writes files to ./output
  python simulate.py --quick    -> smaller/faster settings for a smoke test

Experiment 1 (static)  : compare Nearest Neighbour, 2-opt, Simulated Annealing,
                         Tabu Search and the Genetic Algorithm on random instances.
Experiment 2 (dynamic) : cities are added/removed over time. Compare
                         ADAPTIVE GA (repairs its population) vs RESTART GA
                         (throws everything away on each change).
Output                 : PNG plots, an animated GIF, and results.txt (run log).
"""
import os
import random
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

from ga import GeneticTSP, build_matrix, tour_length
from baselines import (nearest_neighbour, two_opt, simulated_annealing,
                       tabu_search)

QUICK = "--quick" in sys.argv
OUT = "output"
os.makedirs(OUT, exist_ok=True)
LOG = open(os.path.join(OUT, "results.txt"), "w")


def log(*a):
    msg = " ".join(str(x) for x in a)
    print(msg)
    LOG.write(msg + "\n")
    LOG.flush()


def rand_points(n, rng):
    return [(rng.random() * 100, rng.random() * 100) for _ in range(n)]


# ----------------------------------------------------------------------------
# Experiment 1: static comparison
# ----------------------------------------------------------------------------
def experiment_static():
    log("=" * 64)
    log("EXPERIMENT 1: static comparison (mean over instances)")
    log("=" * 64)
    sizes = [30, 60] if QUICK else [30, 60, 100]
    trials = 1 if QUICK else 3
    gens = 150 if QUICK else 400
    names = ["NearestNeighbour", "2-opt(NN)", "SimAnnealing", "TabuSearch", "GeneticAlg"]
    table = {n: {s: [] for s in sizes} for n in names}
    times = {n: {s: [] for s in sizes} for n in names}
    ga_curve = {}

    for n in sizes:
        for trial in range(trials):
            rng = random.Random(1000 * n + trial)
            pts = rand_points(n, rng)
            D = build_matrix(pts)

            def record(name, length, t0):
                table[name][n].append(length)
                times[name][n].append(time.time() - t0)

            t0 = time.time(); t = nearest_neighbour(D)
            record("NearestNeighbour", tour_length(t, D), t0)
            t0 = time.time(); t = two_opt(nearest_neighbour(D), D)
            record("2-opt(NN)", tour_length(t, D), t0)
            t0 = time.time()
            _, L, _ = simulated_annealing(D, iterations=60000 if QUICK else 150000, seed=trial)
            record("SimAnnealing", L, t0)
            t0 = time.time()
            _, L, _ = tabu_search(D, iterations=100 if QUICK else 250, seed=trial)
            record("TabuSearch", L, t0)
            t0 = time.time()
            ga = GeneticTSP(pts, pop_size=80, seed=trial)
            hist = ga.run(gens)
            record("GeneticAlg", ga.best()[1], t0)
            if trial == 0:
                ga_curve[n] = hist

    log(f"{'Algorithm':<18}" + "".join(f"n={s:<6}(len / sec)   " for s in sizes))
    for name in names:
        row = f"{name:<18}"
        for s in sizes:
            L = sum(table[name][s]) / len(table[name][s])
            T = sum(times[name][s]) / len(times[name][s])
            row += f"{L:8.1f} / {T:5.2f}s     "
        log(row)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for n, h in ga_curve.items():
        ax[0].plot(h, label=f"n={n}")
    ax[0].set(title="GA convergence (best tour length per generation)",
              xlabel="Generation", ylabel="Best length")
    ax[0].legend(); ax[0].grid(alpha=.3)
    w = 0.16
    for k, name in enumerate(names):
        vals = [sum(table[name][s]) / len(table[name][s]) for s in sizes]
        ax[1].bar([i + k * w for i in range(len(sizes))], vals, w, label=name)
    ax[1].set_xticks([i + 2 * w for i in range(len(sizes))])
    ax[1].set_xticklabels([f"n={s}" for s in sizes])
    ax[1].set(title="Final tour length (lower is better)", ylabel="Length")
    ax[1].legend(fontsize=8); ax[1].grid(axis="y", alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "exp1_static_comparison.png"), dpi=130)
    plt.close()


# ----------------------------------------------------------------------------
# Experiment 2: dynamic environment
# ----------------------------------------------------------------------------
def make_event_schedule(total_gens, period, seed):
    """Every `period` generations: add 3 cities, remove 1 (a delivery 'cancelled')."""
    rng = random.Random(seed)
    events = {}
    for g in range(period, total_gens, period):
        events[g] = {"add": [(rng.random() * 100, rng.random() * 100) for _ in range(3)],
                     "remove_rand": rng.random()}
    return events


def experiment_dynamic():
    log("")
    log("=" * 64)
    log("EXPERIMENT 2: dynamic cities, adaptive GA vs restart GA")
    log("=" * 64)
    total = 240 if QUICK else 600
    period = 60
    start_n = 120
    pts0 = rand_points(start_n, random.Random(11))
    events = make_event_schedule(total, period, 12)

    LR = 0.03   # light local search so progress is visible over time
    adaptive = GeneticTSP(pts0, pop_size=80, seed=1, local_rate=LR)
    restart = GeneticTSP(pts0, pop_size=80, seed=1, local_rate=LR)
    ref_len = []                      # 2-opt(NN) reference on the current map
    hist_a, hist_r, n_hist, frames = [], [], [], []
    event_gens = []

    for g in range(total):
        if g in events:
            ev = events[g]
            for p in ev["add"]:
                adaptive.add_city(p)
            idx = int(ev["remove_rand"] * len(adaptive.points))
            adaptive.remove_city(idx)
            # Restart GA: same new map, but population thrown away
            new_pts = list(adaptive.points)
            restart = GeneticTSP(new_pts, pop_size=80, seed=g, greedy_seed=False, local_rate=LR)
            event_gens.append(g)
            log(f"gen {g:4d}: +3 cities, -1 city -> now {len(new_pts)} cities")
        adaptive.step(); restart.step()
        ha, hr = adaptive.best()[1], restart.best()[1]
        hist_a.append(ha); hist_r.append(hr); n_hist.append(len(adaptive.points))
        if g % 5 == 0:
            frames.append((g, list(adaptive.points), adaptive.best()[0], ha,
                           g in events))

    # log recovery right after each change
    log("")
    log("Best tour length k generations after each change (lower = recovered faster):")
    log(f"{'gen':>5} {'k':>3} {'adaptive':>10} {'restart':>10}")
    for g in event_gens:
        for k in (1, 10, 40):
            kk = min(g + k, total - 1)
            log(f"{g:5d} {k:3d} {hist_a[kk]:10.1f} {hist_r[kk]:10.1f}")
    log(f"\nFinal: adaptive={hist_a[-1]:.1f}  restart={hist_r[-1]:.1f}  "
        f"cities={n_hist[-1]}")

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(hist_a, label="Adaptive GA (repairs population)")
    ax.plot(hist_r, label="Restart GA (from scratch)", alpha=.8)
    for g in event_gens:
        ax.axvline(g, color="gray", ls=":", lw=.8)
    ax.set(title="Dynamic TSP: cities added/removed every 60 generations",
           xlabel="Generation", ylabel="Best tour length")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "exp2_dynamic_adaptation.png"), dpi=130)
    plt.close()

    # ---- animation ----
    fig, ax = plt.subplots(figsize=(6, 6))

    def draw(i):
        ax.clear()
        g, pts, tour, length, changed = frames[i]
        xs = [pts[c][0] for c in tour] + [pts[tour[0]][0]]
        ys = [pts[c][1] for c in tour] + [pts[tour[0]][1]]
        ax.plot(xs, ys, "-", color="tab:blue", lw=1.3)
        ax.scatter([p[0] for p in pts], [p[1] for p in pts], c="tab:red", s=22, zorder=3)
        ax.set(xlim=(-5, 105), ylim=(-5, 105), xticks=[], yticks=[],
               title=f"Gen {g} | cities {len(pts)} | length {length:.0f}"
                     + ("  <- map changed" if changed else ""))

    anim = FuncAnimation(fig, draw, frames=len(frames), interval=120)
    anim.save(os.path.join(OUT, "dynamic_simulation.gif"), writer=PillowWriter(fps=8))
    plt.close()


if __name__ == "__main__":
    experiment_static()
    experiment_dynamic()
    log("\nDone. Files written to ./output")

In [ ]:
import simulate
simulate.QUICK = FAST
simulate.experiment_static()
display(Image("output/exp1_static_comparison.png"))

**How to read it:** the left plot is the GA's best tour length per generation (most gains happen early because 2-opt refines tours quickly).
The right plot compares final lengths. In the report, the GA was shortest at every size and slowest in time.

## Step 6. Experiment 2: the map changes while running (Table 2, Figure 2)

Start with 120 cities. **Every 60 generations, 3 new cities appear and 1 is removed**, for 600 generations (9 changes).

- **Adaptive GA:** repairs its population (`add_city` / `remove_city`) and keeps evolving.
- **Restart GA:** throws everything away and starts again on the new map.

To compare them, we record the best length *k* generations after each change (k = 1, 10, 40) and compute

`adaptive advantage = (restart - adaptive) / restart`

then average over the 9 changes. A positive number means the adaptive GA is shorter.

In [ ]:
simulate.experiment_dynamic()
display(Image("output/exp2_dynamic_adaptation.png"))
print(open("output/results.txt").read().split("EXPERIMENT 2")[1])

In [ ]:
# Recompute Table 2 yourself from the log lines
import re
lines = open("output/results.txt").read().split("EXPERIMENT 2")[1].splitlines()
rows = [re.findall(r"\d+\.\d+|\d+", l) for l in lines if re.match(r"\s+\d+\s+(1|10|40)\s", l)]
for k in ("1", "10", "40"):
    a = [float(r[2]) for r in rows if r[1] == k]
    r_ = [float(r[3]) for r in rows if r[1] == k]
    if a:
        ma, mr = sum(a) / len(a), sum(r_) / len(r_)
        print(f"{k:>2} generations after change:  adaptive {ma:.1f}  restart {mr:.1f}  advantage {100*(mr-ma)/mr:.1f}%")

**What to notice:** the adaptive GA is far ahead right after a change, the gap shrinks as the restart catches up, and sometimes the
restart finishes ahead. That points to **lost diversity** in the repaired population, a known issue in dynamic optimization.

## Step 7. The Santa Rosa, Laguna scenario

**Real coordinates** (looked up with Google Places): the Laguna Technopark depot, SM City Santa Rosa, Robinsons, Balibago Complex,
Santa Rosa City Hall, Paseo de Santa Rosa, Paseo Outlets, Vista Mall, Ayala Malls Nuvali and QualiMed Hospital.

**Distance:** the *Haversine* formula gives straight-line kilometres between GPS points, multiplied by **1.3** as a rough detour factor.
This is an **assumption**, not real road distance.

**Events:** traffic on the Tagaytay Road side (links x1.6), an express order, a flood near City Hall (links x3), then all clear.
At every phase the result is compared with the exact **Held-Karp** optimum (feasible for 10 or 11 stops).

In [ ]:
%%writefile santa_rosa.py
"""
santa_rosa.py - Hyper-local delivery route optimiser for Santa Rosa, Laguna.

Uses the from-scratch Genetic Algorithm in ga.py on REAL landmark coordinates
(looked up with Google Places) and changes the problem while it runs:

  gen   0 : baseline map, 10 stops
  gen  60 : traffic congestion on the Santa Rosa-Tagaytay Road corridor
  gen 120 : a new EXPRESS order appears (population is repaired, not restarted)
  gen 180 : flooding makes the City Hall area costly to reach
  gen 240 : traffic clears and the flood subsides

At every phase the GA result is compared with the exact optimum (Held-Karp,
feasible because there are only 10-11 stops) so you can see how close it gets.

  python santa_rosa.py            -> writes output/santa_rosa_* files

ASSUMPTIONS (state these in your report):
  * Distance = straight-line (Haversine) x 1.3 detour factor, NOT real road
    distance. Real road networks need a routing API.
  * Congestion / flood multipliers are scenario parameters, not measured data.
  * The express order location is simulated (a point near Balibago Complex).
"""
import math
import os
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

from ga import GeneticTSP, build_matrix, tour_length
from baselines import nearest_neighbour, held_karp

OUT = "output"
os.makedirs(OUT, exist_ok=True)
LOG = open(os.path.join(OUT, "santa_rosa_results.txt"), "w")


def log(*a):
    msg = " ".join(str(x) for x in a)
    print(msg)
    LOG.write(msg + "\n")
    LOG.flush()


# ----------------------------------------------------------------------------
# Real coordinates (lat, lon) from Google Places
# ----------------------------------------------------------------------------
STOPS = [
    ("Laguna Technopark (Depot)", 14.2613265, 121.0575963),
    ("SM City Sta. Rosa",         14.3115473, 121.0985251),
    ("Robinsons Sta. Rosa",       14.3191667, 121.0966667),
    ("Balibago Complex",          14.2915997, 121.0980124),
    ("Santa Rosa City Hall",      14.3146492, 121.1124535),
    ("Paseo de Santa Rosa",       14.2477814, 121.0631716),
    ("Paseo Outlets (Greenfield)", 14.2485967, 121.0643075),
    ("Vista Mall Sta. Rosa",      14.2434182, 121.0580339),
    ("Ayala Malls Nuvali",        14.2367513, 121.0577278),
    ("QualiMed Hospital",         14.2321549, 121.0528732),
]
EXPRESS = ("EXPRESS order (Balibago, simulated)", 14.2938, 121.1010)

DETOUR = 1.3                      # straight-line -> road distance (assumption)
CORRIDOR = {"Paseo de Santa Rosa", "Paseo Outlets (Greenfield)", "Vista Mall Sta. Rosa",
            "Ayala Malls Nuvali", "QualiMed Hospital"}   # Tagaytay Rd side
CONGESTION = 1.6                  # cost multiplier on links touching the corridor
FLOOD_NODE = "Santa Rosa City Hall"
FLOOD = 3.0                       # cost multiplier on links touching it


def haversine_km(a, b):
    """Great-circle distance in km between (lat, lon) points."""
    R = 6371.0
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    h = (math.sin((la2 - la1) / 2) ** 2
         + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(h))


def road_km(a, b):
    return haversine_km(a, b) * DETOUR


class Scenario:
    """Holds the names/points and the currently active conditions."""
    def __init__(self):
        self.names = [s[0] for s in STOPS]
        self.points = [(s[1], s[2]) for s in STOPS]
        self.traffic = False
        self.flood = False

    def cost_matrix(self):
        D = build_matrix(self.points, road_km)
        n = len(D)
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                f = 1.0
                if self.traffic and (self.names[i] in CORRIDOR or self.names[j] in CORRIDOR):
                    f *= CONGESTION
                if self.flood and FLOOD_NODE in (self.names[i], self.names[j]):
                    f *= FLOOD
                D[i][j] *= f
        return D


def main():
    sc = Scenario()
    ga = GeneticTSP(sc.points, pop_size=80, seed=7, dist_fn=road_km, local_rate=0.05)
    ga.set_distances(sc.cost_matrix())

    events = {
        60:  ("TRAFFIC: Tagaytay-Sta. Rosa Rd congested (x%.1f cost)" % CONGESTION, "traffic_on"),
        120: ("NEW ORDER: express delivery near Balibago", "add"),
        180: ("FLOOD: City Hall area costly (x%.1f cost)" % FLOOD, "flood_on"),
        240: ("CLEARED: traffic and flood back to normal", "clear"),
    }
    total = 300
    hist, frames, markers, summary = [], [], [], []

    def snapshot(label):
        D = ga.D
        _, L = ga.best()
        nn = tour_length(nearest_neighbour(D), D)
        opt = held_karp(D)
        summary.append((label, len(ga.points), L, opt, nn))
        log(f"  {label:<44} stops={len(ga.points):2d}  GA={L:7.2f}  optimal={opt:7.2f}  "
            f"gap={max(0.0, 100 * (L - opt) / opt):5.2f}%  nearest-neighbour={nn:7.2f}")

    log("=" * 100)
    log("SANTA ROSA, LAGUNA - dynamic delivery routing with a Genetic Algorithm")
    log("Costs are effective km (straight-line x 1.3 detour x scenario multipliers)")
    log("=" * 100)
    t0 = time.time()
    for g in range(total):
        if g in events:
            text, kind = events[g]
            snapshot("before: " + ("start" if g == 0 else text.split(":")[0].lower()))
            if kind == "traffic_on":
                sc.traffic = True
            elif kind == "flood_on":
                sc.flood = True
            elif kind == "clear":
                sc.traffic = sc.flood = False
            elif kind == "add":
                sc.names.append(EXPRESS[0])
                sc.points.append((EXPRESS[1], EXPRESS[2]))
                ga.add_city(sc.points[-1])
            ga.set_distances(sc.cost_matrix())
            markers.append((g, text))
            log(f"gen {g:3d}: {text}")
        ga.step()
        _, L = ga.best()
        hist.append(L)
        if g % 4 == 0:
            frames.append((g, list(sc.points), list(sc.names), ga.best()[0], L,
                           sc.traffic, sc.flood,
                           next((m[1] for m in markers if 0 <= g - m[0] < 24), "")))
    snapshot("end of run (all clear)")
    log(f"\nTotal evolution time: {time.time() - t0:.1f} s for {total} generations")

    # best final route
    tour, L = ga.best()
    start = tour.index(0)
    tour = tour[start:] + tour[:start]
    log("\nFinal route (depot first):")
    for i in tour:
        log(f"  -> {sc.names[i]}")
    log(f"  -> {sc.names[0]} (return)   total = {L:.2f} effective km")

    # ---------------- plots ----------------
    def draw_map(ax, pts, names, tr, length, traffic=False, flood=False, title=""):
        lat0 = math.radians(14.28)
        ax.set_aspect(1 / math.cos(lat0))
        if traffic:
            cx = [p[1] for p, n in zip(pts, names) if n in CORRIDOR]
            cy = [p[0] for p, n in zip(pts, names) if n in CORRIDOR]
            ax.scatter(cx, cy, s=900, c="orange", alpha=.25, zorder=1)
        xs = [pts[c][1] for c in tr] + [pts[tr[0]][1]]
        ys = [pts[c][0] for c in tr] + [pts[tr[0]][0]]
        ax.plot(xs, ys, "-", color="tab:blue", lw=1.8, zorder=2)
        for k, (p, n) in enumerate(zip(pts, names)):
            col = "tab:green" if k == 0 else ("magenta" if n.startswith("EXPRESS") else "tab:red")
            if flood and n == FLOOD_NODE:
                col = "navy"
            ax.scatter(p[1], p[0], c=col, s=70 if k else 140, marker="*" if k == 0 else "o", zorder=3)
            ax.annotate(n.replace(" (Depot)", ""), (p[1], p[0]), fontsize=6.5,
                        xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.set_title(title, fontsize=10)
        ax.grid(alpha=.3)

    fig, ax = plt.subplots(figsize=(7.5, 8))
    draw_map(ax, sc.points, sc.names, tour, L, title=f"Optimised delivery route, Santa Rosa, Laguna ({L:.1f} effective km)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "santa_rosa_route.png"), dpi=140); plt.close()

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(hist, color="tab:blue")
    for g, text in markers:
        ax.axvline(g, color="gray", ls=":")
        ax.text(g + 2, max(hist) * 0.995, text.split(":")[0], rotation=90, va="top", fontsize=8)
    ax.set(title="GA best route cost over time (cost jumps are real: the world changed)",
           xlabel="Generation", ylabel="Effective km")
    ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "santa_rosa_timeline.png"), dpi=140); plt.close()

    fig, ax = plt.subplots(figsize=(7, 7.5))

    def draw(i):
        ax.clear()
        g, pts, names, tr, length, traffic, flood, note = frames[i]
        draw_map(ax, pts, names, tr, length, traffic, flood,
                 title=f"Gen {g} | stops {len(pts)} | cost {length:.1f} km" + (f"\n{note}" if note else ""))

    anim = FuncAnimation(fig, draw, frames=len(frames), interval=150)
    anim.save(os.path.join(OUT, "santa_rosa_simulation.gif"), writer=PillowWriter(fps=7))
    plt.close()
    log("\nSaved: santa_rosa_route.png, santa_rosa_timeline.png, santa_rosa_simulation.gif")


if __name__ == "__main__":
    main()

In [ ]:
import santa_rosa
santa_rosa.main()
display(Image("output/santa_rosa_route.png"))
display(Image("output/santa_rosa_timeline.png"))

Open `output/santa_rosa_simulation.gif` in the Colab file browser (folder icon on the left) to see the animation.
**Reading the log:** `gap` is how far the GA is above the exact optimum; `nearest-neighbour` is the greedy route's cost. The GA matched the
optimum in every phase, while greedy was much worse during the flood.

## Step 8. Use case scenarios with equal time budgets (Section 8, Figure 3)

Every algorithm gets the **same wall-clock time** in each scenario.

| Scenario | Setup | Budget | Metric |
|---|---|---|---|
| S1 Small daily route | 10 Santa Rosa stops | 50 ms | % above the exact optimum |
| S2 Large overnight batch | 150 random stops | 6 s | mean tour length |
| S3 Real-time dispatch | 60 and 150 random stops | 250 ms | mean tour length |
| S4 Orders arrive and cancel | 60 stops, +3 / -1 per change, 6 changes | 400 ms per change | mean % above the best method at that change |
| S5 Traffic and flooding | 10 Santa Rosa stops, 4 cost changes | 150 ms per change | % above the exact optimum |

Methods: Genetic Algorithm, Simulated Annealing, Tabu Search, plus Nearest Neighbour and 2-opt as instant references.
**"Warm"** means the method continues from its previous tour (repaired after a change); **"restart"** means it starts again from scratch.

The formulas are simple. For a method with cost *c*: **gap to optimum = 100 x (c - optimum) / optimum**, and
**% above best = 100 x (c - best of all methods) / best of all methods**.

In [ ]:
%%writefile scenarios.py
"""
scenarios.py - Use case scenarios that compare the algorithms.

Every algorithm gets the SAME wall-clock budget in each scenario (equal-time
comparison), so differences come from the algorithm, not from how long it ran.

  S1  Small daily route      : 10 Santa Rosa stops, tiny budget (50 ms)
  S2  Large overnight batch  : 150 stops, generous budget (6 s)
  S3  Real-time dispatch     : 60 and 150 stops, strict budget (250 ms)
  S4  Orders arrive/cancel   : stops added and removed, budget per change (400 ms)
  S5  Traffic and flooding   : same stops, costs change, budget per change (150 ms)

  python scenarios.py            -> full run (~2 min), writes output/use_case_*
  python scenarios.py --quick    -> shorter budgets, smoke test

Algorithms: Genetic Algorithm (ga.py), Simulated Annealing, Tabu Search,
plus Nearest Neighbour and 2-opt as instant references. 'warm' means the
algorithm continues from its previous tour (repaired after a change);
'restart' means it starts again from scratch.
"""
import math
import os
import random
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ga import GeneticTSP, build_matrix, tour_length
from baselines import nearest_neighbour, two_opt, held_karp
import santa_rosa as sr

QUICK = "--quick" in sys.argv
Q = 0.25 if QUICK else 1.0
OUT = "output"
os.makedirs(OUT, exist_ok=True)
LOG = open(os.path.join(OUT, "use_case_results.txt"), "w")
now = time.perf_counter


def log(*a):
    msg = " ".join(str(x) for x in a)
    print(msg)
    LOG.write(msg + "\n")
    LOG.flush()


def mean(x):
    return sum(x) / len(x)


# ----------------------------------------------------------------------------
# Time-budgeted versions of the algorithms
# ----------------------------------------------------------------------------
def sa_timed(D, start, budget, seed=0, t0_scale=1.0):
    """Simulated annealing; temperature falls geometrically over the budget."""
    rng = random.Random(seed)
    n = len(D)
    tour = start[:]
    cur = tour_length(tour, D)
    best, bl = tour[:], cur
    T0 = t0_scale * cur / n
    T = T0
    t_start, it = now(), 0
    while True:
        if it % 200 == 0:
            frac = (now() - t_start) / budget
            if frac >= 1:
                break
            T = max(T0 * (1e-3 ** frac), 1e-12)
        it += 1
        i, j = sorted(rng.sample(range(n), 2))
        if i == 0 and j == n - 1:
            continue
        a, b = tour[i - 1], tour[i]
        c, d = tour[j], tour[(j + 1) % n]
        delta = D[a][c] + D[b][d] - D[a][b] - D[c][d]
        if delta < 0 or rng.random() < math.exp(-delta / T):
            tour[i:j + 1] = reversed(tour[i:j + 1])
            cur += delta
            if cur < bl - 1e-12:
                bl, best = cur, tour[:]
    return best, tour_length(best, D)


def tabu_timed(D, start, budget, seed=0, tenure=15, sample=100):
    """Tabu search over sampled 2-opt moves with edge tabu list + aspiration."""
    rng = random.Random(seed)
    n = len(D)
    tour = start[:]
    cur = tour_length(tour, D)
    best, bl = tour[:], cur
    tabu, it, t_start = {}, 0, now()
    while now() - t_start < budget:
        move, md = None, float("inf")
        for _ in range(sample):
            i, j = sorted(rng.sample(range(n), 2))
            if i == 0 and j == n - 1:
                continue
            a, b = tour[i - 1], tour[i]
            c, d = tour[j], tour[(j + 1) % n]
            delta = D[a][c] + D[b][d] - D[a][b] - D[c][d]
            key = (min(a, c), max(a, c))
            if tabu.get(key, -1) > it and cur + delta >= bl:
                continue
            if delta < md:
                move, md = (i, j, key), delta
        it += 1
        if move is None:
            continue
        i, j, key = move
        tour[i:j + 1] = reversed(tour[i:j + 1])
        cur += md
        tabu[key] = it + tenure
        if cur < bl - 1e-12:
            bl, best = cur, tour[:]
    return best, tour_length(best, D)


def run_ga(ga, budget, t_start=None):
    """Evolve until the budget is used (always at least one generation)."""
    t0 = t_start if t_start is not None else now()
    while True:
        ga.step()
        if now() - t0 >= budget:
            break
    return ga.best()[1]


def insert_city(tour, D, new):
    bp, bc = 0, float("inf")
    for p in range(len(tour)):
        a, b = tour[p - 1], tour[p]
        c = D[a][new] + D[new][b] - D[a][b]
        if c < bc:
            bc, bp = c, p
    t = tour[:]
    t.insert(bp, new)
    return t


def remove_city(tour, idx):
    return [c - 1 if c > idx else c for c in tour if c != idx]


def rand_points(n, rng):
    return [(rng.random() * 100, rng.random() * 100) for _ in range(n)]


RESULTS = {}


# ----------------------------------------------------------------------------
# S1: small daily route (Santa Rosa, 10 stops)
# ----------------------------------------------------------------------------
def s1():
    B = 0.05 * Q
    sc = sr.Scenario()
    D = sc.cost_matrix()
    opt = held_karp(D)
    nn = nearest_neighbour(D)
    names = ["Nearest Neighbour", "2-opt", "Simulated Annealing", "Tabu Search", "Genetic Algorithm"]
    gaps = {k: [] for k in names}
    for seed in range(5):
        gaps["Nearest Neighbour"].append(tour_length(nn, D))
        gaps["2-opt"].append(tour_length(two_opt(nn, D), D))
        gaps["Simulated Annealing"].append(sa_timed(D, nn, B, seed)[1])
        gaps["Tabu Search"].append(tabu_timed(D, nn, B, seed)[1])
        t0 = now()
        ga = GeneticTSP(sc.points, pop_size=60, seed=seed, dist_fn=sr.road_km)
        ga.set_distances(D)
        gaps["Genetic Algorithm"].append(run_ga(ga, B, t0))
    log("=" * 78)
    log(f"S1 SMALL DAILY ROUTE - 10 Santa Rosa stops, {B * 1000:.0f} ms budget, 5 seeds")
    log(f"   exact optimum (Held-Karp) = {opt:.2f} effective km")
    out = {}
    for k in names:
        g = max(0.0, 100 * (mean(gaps[k]) - opt) / opt)
        out[k] = g
        log(f"   {k:<20} mean cost {mean(gaps[k]):7.2f}   gap to optimum {g:5.2f}%")
    RESULTS["S1"] = out


# ----------------------------------------------------------------------------
# S2: large overnight batch (150 stops)
# ----------------------------------------------------------------------------
def s2():
    B = 6.0 * Q
    n, trials = 150, 2
    names = ["Nearest Neighbour", "2-opt", "Simulated Annealing", "Tabu Search", "Genetic Algorithm"]
    res = {k: [] for k in names}
    for tr in range(trials):
        pts = rand_points(n, random.Random(500 + tr))
        D = build_matrix(pts)
        nn = nearest_neighbour(D)
        res["Nearest Neighbour"].append(tour_length(nn, D))
        res["2-opt"].append(tour_length(two_opt(nn, D), D))
        res["Simulated Annealing"].append(sa_timed(D, nn, B, tr)[1])
        res["Tabu Search"].append(tabu_timed(D, nn, B, tr)[1])
        t0 = now()
        ga = GeneticTSP(pts, pop_size=80, seed=tr)
        res["Genetic Algorithm"].append(run_ga(ga, B, t0))
    log("=" * 78)
    log(f"S2 LARGE OVERNIGHT BATCH - {n} stops, {B:.1f} s budget, {trials} instances")
    out = {}
    for k in names:
        out[k] = mean(res[k])
        log(f"   {k:<20} mean tour length {out[k]:8.1f}")
    RESULTS["S2"] = out


# ----------------------------------------------------------------------------
# S3: real-time dispatch (strict budget)
# ----------------------------------------------------------------------------
def s3():
    B = 0.25 * Q
    names = ["Nearest Neighbour", "2-opt", "Simulated Annealing", "Tabu Search", "Genetic Algorithm"]
    log("=" * 78)
    log(f"S3 REAL-TIME DISPATCH - strict {B * 1000:.0f} ms budget, 5 instances each size")
    RESULTS["S3"] = {}
    for n in (60, 150):
        res = {k: [] for k in names}
        gens = []
        for tr in range(5):
            pts = rand_points(n, random.Random(900 + n + tr))
            D = build_matrix(pts)
            nn = nearest_neighbour(D)
            res["Nearest Neighbour"].append(tour_length(nn, D))
            res["2-opt"].append(tour_length(two_opt(nn, D), D))
            res["Simulated Annealing"].append(sa_timed(D, nn, B, tr)[1])
            res["Tabu Search"].append(tabu_timed(D, nn, B, tr)[1])
            t0 = now()
            ga = GeneticTSP(pts, pop_size=80, seed=tr)
            res["Genetic Algorithm"].append(run_ga(ga, B, t0))
            gens.append(ga.generation)
        log(f"   n = {n}  (GA completed about {mean(gens):.0f} generations)")
        RESULTS["S3"][n] = {}
        for k in names:
            RESULTS["S3"][n][k] = mean(res[k])
            log(f"      {k:<20} mean tour length {mean(res[k]):8.1f}")


# ----------------------------------------------------------------------------
# S4: orders arrive and cancel (stops added / removed)
# ----------------------------------------------------------------------------
def s4():
    B = 0.4 * Q
    events, insts = 6, 3
    names = ["GA adaptive", "GA restart", "SA warm", "SA restart", "Tabu warm", "Tabu restart"]
    above = {k: [] for k in names}
    for inst in range(insts):
        rng = random.Random(2000 + inst)
        pts = rand_points(60, rng)
        D = build_matrix(pts)
        nn = nearest_neighbour(D)
        ga_ad = GeneticTSP(pts, pop_size=80, seed=inst, local_rate=0.05)
        run_ga(ga_ad, B)
        warm_sa, warm_tb = sa_timed(D, nn, B, inst)[0], tabu_timed(D, nn, B, inst)[0]
        for ev in range(events):
            for _ in range(3):                       # 3 new orders
                p = (rng.random() * 100, rng.random() * 100)
                pts.append(p)
                D = build_matrix(pts)
                new = len(pts) - 1
                ga_ad.add_city(p)
                warm_sa = insert_city(warm_sa, D, new)
                warm_tb = insert_city(warm_tb, D, new)
            idx = rng.randrange(1, len(pts))         # 1 cancelled order
            del pts[idx]
            D = build_matrix(pts)
            ga_ad.remove_city(idx)
            warm_sa, warm_tb = remove_city(warm_sa, idx), remove_city(warm_tb, idx)
            nn = nearest_neighbour(D)
            cost = {}
            cost["GA adaptive"] = run_ga(ga_ad, B)
            t0 = now()
            g2 = GeneticTSP(pts, pop_size=80, seed=ev, local_rate=0.05)
            cost["GA restart"] = run_ga(g2, B, t0)
            warm_sa, cost["SA warm"] = sa_timed(D, warm_sa, B, ev, t0_scale=0.1)
            cost["SA restart"] = sa_timed(D, nn, B, ev)[1]
            warm_tb, cost["Tabu warm"] = tabu_timed(D, warm_tb, B, ev)
            cost["Tabu restart"] = tabu_timed(D, nn, B, ev)[1]
            bestk = min(cost.values())
            for k in names:
                above[k].append(100 * (cost[k] - bestk) / bestk)
    log("=" * 78)
    log(f"S4 ORDERS ARRIVE AND CANCEL - start 60 stops, {events} changes (+3 / -1 stops), "
        f"{B * 1000:.0f} ms per change, {insts} instances")
    log("   Mean % above the best tour found by any method at that change (0 = best):")
    out = {}
    for k in names:
        out[k] = mean(above[k])
        log(f"   {k:<14} {out[k]:6.2f}%")
    RESULTS["S4"] = out


# ----------------------------------------------------------------------------
# S5: traffic and flooding (same stops, changing costs) - Santa Rosa
# ----------------------------------------------------------------------------
def s5():
    B = 0.15 * Q
    phases = [("normal", False, False), ("traffic", True, False),
              ("flood", True, True), ("cleared", False, False)]
    names = ["Nearest Neighbour", "2-opt", "GA adaptive", "GA restart",
             "SA warm", "SA restart", "Tabu warm", "Tabu restart"]
    gaps = {k: [] for k in names}
    per_phase = {k: {p[0]: [] for p in phases} for k in names}
    for seed in range(5):
        sc = sr.Scenario()
        ga = GeneticTSP(sc.points, pop_size=60, seed=seed, dist_fn=sr.road_km, local_rate=0.05)
        warm_sa = warm_tb = list(range(len(sc.points)))
        for name, traffic, flood in phases:
            sc.traffic, sc.flood = traffic, flood
            D = sc.cost_matrix()
            opt = held_karp(D)
            nn = nearest_neighbour(D)
            cost = {"Nearest Neighbour": tour_length(nn, D),
                    "2-opt": tour_length(two_opt(nn, D), D)}
            ga.set_distances(D)
            cost["GA adaptive"] = run_ga(ga, B)
            t0 = now()
            g2 = GeneticTSP(sc.points, pop_size=60, seed=seed, dist_fn=sr.road_km, local_rate=0.05)
            g2.set_distances(D)
            cost["GA restart"] = run_ga(g2, B, t0)
            warm_sa, cost["SA warm"] = sa_timed(D, warm_sa, B, seed, t0_scale=0.1)
            cost["SA restart"] = sa_timed(D, nn, B, seed)[1]
            warm_tb, cost["Tabu warm"] = tabu_timed(D, warm_tb, B, seed)
            cost["Tabu restart"] = tabu_timed(D, nn, B, seed)[1]
            for k in names:
                g = max(0.0, 100 * (cost[k] - opt) / opt)
                gaps[k].append(g)
                per_phase[k][name].append(g)
    log("=" * 78)
    log(f"S5 TRAFFIC AND FLOODING - 10 Santa Rosa stops, 4 cost changes, "
        f"{B * 1000:.0f} ms per change, 5 seeds")
    log("   Mean % above the exact optimum (Held-Karp), 0 = optimal:")
    out = {}
    for k in names:
        out[k] = mean(gaps[k])
        detail = "  ".join(f"{p[0]} {mean(per_phase[k][p[0]]):5.2f}%" for p in phases)
        log(f"   {k:<18} overall {out[k]:6.2f}%   ({detail})")
    RESULTS["S5"] = out


# ----------------------------------------------------------------------------
def plots():
    fig, ax = plt.subplots(2, 3, figsize=(15, 8))
    cols = {"Nearest Neighbour": "#999999", "2-opt": "#c9a227", "Simulated Annealing": "#4c72b0",
            "Tabu Search": "#dd8452", "Genetic Algorithm": "#55a868"}

    def bars(a, d, title, ylabel, colmap=None):
        keys = list(d)
        a.bar(range(len(keys)), [d[k] for k in keys],
              color=[(colmap or {}).get(k, "#4c72b0") for k in keys])
        a.set_xticks(range(len(keys)))
        a.set_xticklabels(keys, rotation=35, ha="right", fontsize=8)
        a.set_title(title, fontsize=10)
        a.set_ylabel(ylabel)
        a.grid(axis="y", alpha=.3)

    bars(ax[0][0], RESULTS["S1"], "S1 Small daily route", "% above optimum", cols)
    bars(ax[0][1], RESULTS["S2"], "S2 Large batch (150 stops)", "tour length", cols)
    d3 = RESULTS["S3"][150]
    bars(ax[0][2], d3, "S3 Real-time dispatch (150 stops)", "tour length", cols)
    c4 = {k: ("#55a868" if k.startswith("GA") else "#4c72b0" if k.startswith("SA") else "#dd8452") for k in RESULTS["S4"]}
    bars(ax[1][0], RESULTS["S4"], "S4 Orders arrive/cancel", "% above best", c4)
    c5 = {k: ("#55a868" if k.startswith("GA") else "#4c72b0" if k.startswith("SA") else
              "#dd8452" if k.startswith("Tabu") else "#999999") for k in RESULTS["S5"]}
    bars(ax[1][1], RESULTS["S5"], "S5 Traffic and flooding", "% above optimum", c5)
    d3b = RESULTS["S3"][60]
    bars(ax[1][2], d3b, "S3 Real-time dispatch (60 stops)", "tour length", cols)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "use_case_comparison.png"), dpi=130)
    plt.close()


if __name__ == "__main__":
    t0 = now()
    for f in (s1, s2, s3, s4, s5):
        f()
    plots()
    log("=" * 78)
    log(f"Done in {now() - t0:.0f} s. Saved output/use_case_comparison.png")

In [ ]:
import scenarios
scenarios.Q = 0.3 if FAST else 1.0      # scales every time budget
scenarios.s1()

**S1.** With only 10 stops, every metaheuristic should reach (or nearly reach) 0.00% above the optimum.

In [ ]:
scenarios.s2()

**S2.** A large batch with plenty of time. Expect the GA to be shortest, SA close behind, and Tabu Search with sampled moves clearly behind.

In [ ]:
scenarios.s3()

**S3.** Very little time. The GA completes only a handful of generations at 150 stops, so its lead comes largely from its **greedy starting tour**
and **2-opt refinement**, not from evolution alone. That is an honest caveat in the report.

In [ ]:
scenarios.s4()

**S4.** Orders arrive and cancel. Compare *adaptive* vs *restart*. In the report, restarting matched or beat repairing once each change allowed a
few hundred milliseconds, even though repair wins in the first generations after a change.

In [ ]:
scenarios.s5()

**S5.** Costs change with traffic and flooding. Metaheuristics stay near-optimal; the greedy tour can be far off when a flood raises costs.

In [ ]:
scenarios.plots()
display(Image("output/use_case_comparison.png"))

### Use cases (formal format) and their measured results

A **use case** describes a real person trying to reach a goal. Each one below maps to a scenario you just ran.

| ID | Actor | Goal | Input and conditions | Expected outcome (hypothesis) |
|---|---|---|---|---|
| UC-01 | Dispatcher | Plan the small daily route quickly | 10 Santa Rosa stops, 50 ms | Any good method is near-optimal, so the simplest is enough |
| UC-02 | Logistics planner | Plan an overnight batch of deliveries | 150 stops, 6 s | The GA gives the shortest route |
| UC-03 | Real-time dispatcher | Get a usable route immediately | 60 and 150 stops, 250 ms | GA or 2-opt stay competitive when time is short |
| UC-04 | Dispatcher | Keep the route good as orders arrive and cancel | +3 / -1 stops per change, 400 ms | GA stays near the best; SA and Tabu fall behind |
| UC-05 | Dispatcher in bad weather | Re-plan when traffic and flooding raise costs | 10 stops, costs x1.6 and x3, 150 ms | Metaheuristics stay near-optimal; a greedy route fails |

In [ ]:
def winners(d):
    m = min(d.values()); tol = max(0.01, abs(m) * 1e-4)
    return ", ".join(k for k, v in d.items() if v <= m + tol)

R = scenarios.RESULTS
rows = [("UC-01", "S1  % above optimum",      R["S1"]),
        ("UC-02", "S2  mean tour length",    R["S2"]),
        ("UC-03", "S3  length, 60 stops",    R["S3"][60]),
        ("UC-03", "S3  length, 150 stops",   R["S3"][150]),
        ("UC-04", "S4  % above best",        R["S4"]),
        ("UC-05", "S5  % above optimum",     R["S5"])]
print(f"{'Use case':<8}{'Measured on':<26}{'Best method(s)':<80}Worst method")
print("-" * 130)
for uc_id, what, d in rows:
    worst = max(d, key=d.get)
    print(f"{uc_id:<8}{what:<26}{winners(d):<80}{worst} ({d[worst]:.1f})")

Compare this table with the **Expected outcome** column. If a measured result contradicts a hypothesis, that is a finding worth reporting,
not an error. For example, UC-04 is where restarting the GA matched or beat repairing it.

## Reading the results honestly

- **Pure Python, small samples.** Each scenario uses 2 to 5 instances, with budgets I chose. The results show **tendencies, not guarantees**.
- **Hardware matters.** Time-budget results change with your CPU. Run a scenario twice and you will see small differences.
- **Assumptions.** Distances are straight-line x 1.3; traffic and flood multipliers are scenario parameters, not measured data.
- **The GA is memetic.** Its strong results under tight budgets come partly from the greedy seed and 2-opt step.

## Try it yourself (exercises)

1. In `ga.py`, set `local_rate=0` and re-run Step 5. How much does the 2-opt step matter?
2. Change `pop_size` from 80 to 20 and to 300. What happens to quality and time?
3. In Step 6, change the change period from 60 to 20 generations. Does repair help more or less?
4. **Add random immigrants:** after each map change, replace 20% of the population with new random tours. Does the restart advantage disappear?
5. Replace the Haversine x 1.3 distance with a real driving-distance matrix for the 10 Santa Rosa stops.

## References (verified)

- Larranaga, P., et al. (1999). Genetic algorithms for the travelling salesman problem: A review of representations and operators. *Artificial Intelligence Review, 13*(2), 129-170.
- Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983). Optimization by simulated annealing. *Science, 220*(4598), 671-680.
- Glover, F. (1989). Tabu search, Part I. *ORSA Journal on Computing, 1*(3), 190-206.
- Ahmed, Z. H., Haron, H., & Al-Tameem, A. (2024). Appropriate combination of crossover operator and mutation operator in genetic algorithms for the travelling salesman problem. *Computers, Materials & Continua, 79*(2), 2399-2425.
- Ilhan, I., & Gokmen, G. (2022). A list-based simulated annealing algorithm with crossover operator for the traveling salesman problem. *Neural Computing and Applications, 34*(10), 7627-7652.
- Werth, B., Karder, J., Wagner, S., & Affenzeller, M. (2025). Diversity management in evolutionary dynamic optimization. In *EUROCAST 2024* (LNCS 15172, pp. 140-147).